### D-MPNN traing using chemprop v2

In [1]:
from pathlib import Path
import pandas as pd
import mlflow 
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
import chemprop
from chemprop import data, featurizers, models, nn


### 1. Load scaffold split

In [2]:
data_path = Path.cwd().parent
splits_dir = data_path / "data/splits"

In [3]:
smiles_column = "smiles"
target_column = "pIC50"
train_df = pd.read_csv(splits_dir/"scaffold_train.csv")
val_df = pd.read_csv(splits_dir/"scaffold_val.csv")
test_df = pd.read_csv(splits_dir/"scaffold_test.csv")

train_df.shape, val_df.shape, test_df.shape

((8401, 5), (1050, 5), (1051, 5))

### 2. Build Molecule Datapoints, MoleculeDataset and DataLoader

In [5]:
def df_to_datapoints(df, smiles_col, target_col):
    smiles = df[smiles_col].values
    target = df[[target_col]].values
    return [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(smiles, target)]

train_data = df_to_datapoints(train_df, smiles_column, target_column)
val_data = df_to_datapoints(val_df, smiles_column, target_column)
test_data = df_to_datapoints(test_df, smiles_column, target_column)


In [6]:
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
train_dataset = data.MoleculeDataset(train_data, featurizer)
scaler = train_dataset.normalize_targets()

val_dataset = data.MoleculeDataset(val_data, featurizer)
val_dataset.normalize_targets(scaler)

test_dataset = data.MoleculeDataset(test_data, featurizer)

train_loader = data.build_dataloader(train_dataset, num_workers=0)
val_loader = data.build_dataloader(val_dataset, num_workers = 0, shuffle = False)
test_loader = data.build_dataloader(test_dataset, num_workers = 0, shuffle = False)


In [7]:
train_loader


In [8]:
batch = next(iter(train_loader))
print(type(batch))
print(batch._fields)  # named fields in the batch

<class 'chemprop.data.collate.TrainingBatch'>
('bmg', 'V_d', 'X_d', 'Y', 'w', 'lt_mask', 'gt_mask')


In [9]:
bmg, V_d, X_d, Y, weights, lt_mask, gt_mask = batch

print("BatchMolGraph:", bmg)
print("Y shape:", Y.shape)        # (batch_size, 1) — your pIC50 targets, scaled
print("Y sample:", Y[:5])
print("weights:", weights[:5])    # sample weights, usually all 1s unless you set them

BatchMolGraph: <chemprop.data.collate.BatchMolGraph object at 0x136c09e90>
Y shape: torch.Size([64, 1])
Y sample: tensor([[-0.9808],
        [-1.1055],
        [ 0.8349],
        [ 0.5158],
        [ 0.2590]])
weights: tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.]])


### 3. Build the MPNN

In [10]:
# Message passing
mp = nn.BondMessagePassing()

# aggregation
agg = nn.MeanAggregation()

# Output transform to un-scale predictions back to real pIC50 units
output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)

# Regression FFN
ffn = nn.RegressionFFN(output_transform=output_transform)

# Metrics tracked during training/val
metric_list = [nn.metrics.RMSE(), nn.metrics.MAE()]

batch_norm = True

mpnn = models.MPNN(mp, agg, ffn, batch_norm, metric_list)
mpnn

MPNN(
  (message_passing): BondMessagePassing(
    (W_i): Linear(in_features=86, out_features=300, bias=False)
    (W_h): Linear(in_features=300, out_features=300, bias=False)
    (W_o): Linear(in_features=372, out_features=300, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (tau): ReLU()
    (V_d_transform): Identity()
    (graph_transform): Identity()
  )
  (agg): MeanAggregation()
  (bn): BatchNorm1d(300, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (predictor): RegressionFFN(
    (ffn): MLP(
      (0): Sequential(
        (0): Linear(in_features=300, out_features=300, bias=True)
      )
      (1): Sequential(
        (0): ReLU()
        (1): Dropout(p=0.0, inplace=False)
        (2): Linear(in_features=300, out_features=1, bias=True)
      )
    )
    (criterion): MSE(task_weights=[[1.0]])
    (output_transform): UnscaleTransform()
  )
  (X_d_transform): Identity()
  (metrics): ModuleList(
    (0): RMSE(task_weights=[[1.0]])
    (1): MAE

### 4. Set up and run the trainer

In [11]:
checkpointing = ModelCheckpoint(
    "models/chemprop_checkpoints",
    "best-{epoch}-{val_loss:.2f}",
    "val_loss",
    mode="min",
    save_last=True,
)

trainer = pl.Trainer(
    logger=False,
    enable_checkpointing=True,
    enable_progress_bar=True,
    accelerator="auto",
    devices=1,
    max_epochs=50,   # small run first, just to confirm the loop works end-to-end
    callbacks=[checkpointing],
)



GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [16]:
trainer.fit(mpnn, train_loader, val_loader)

Loading `train_dataloader` to estimate number of stepping batches.
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1.276                                                                      
Modules in train mode: 25                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:4
34: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=5` reached.


In [13]:
with mlflow.start_run(run_name="chemprop_baseline_scaffold_split"):
    mlflow.log_params({
        "split_type": "scaffold",
        "descriptor_type": "learned_graph",
        "model": "chemprop_MPNN_regression",
        "message_passing": "BondMessagePassing",
        "aggregation": "mean",
        "batch_norm": True,
        "max_epochs": 50,
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "test_size": len(test_dataset),
    })

    trainer.fit(mpnn, train_loader, val_loader)

    val_results = trainer.validate(dataloaders=val_loader, weights_only=False)[0]
    mlflow.log_metrics({f"val_{k}": v for k, v in val_results.items()})
    mlflow.log_artifact(str(checkpointing.best_model_path))

/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints exists and is not empty.
Loading `train_dataloader` to estimate number of stepping batches.
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation    │      0 │ train │     0 │
│ 2 │ bn              │ BatchNorm1d        │    600 │ train │     0 │
│ 3 │ predictor       │ RegressionFFN      │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList         │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1.276                                                                      
Modules in train mode: 25                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:4
34: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=50` reached.


/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:149: `.validate(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.validate(ckpt_path='best')` to use the best model or `.validate(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=30-val_loss=0.51.ckpt
Loaded model weights from the checkpoint at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=30-val_loss=0.51.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/mae          │    0.5338643789291382     │
│         val/rmse          │    0.7123720645904541     │
│         val_loss          │    0.5074739456176758     │
└───────────────────────────┴───────────────────────────┘

In [14]:
test_results = trainer.test(dataloaders=test_loader, ckpt_path="best", weights_only=False)[0]
print(test_results)

Restoring states from the checkpoint path at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=30-val_loss=0.51.ckpt
Loaded model weights from the checkpoint at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=30-val_loss=0.51.ckpt


Output()

/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/mae          │    0.7308712601661682     │
│         test/rmse         │    0.9833884239196777     │
└───────────────────────────┴───────────────────────────┘

{'test/rmse': 0.9833884239196777, 'test/mae': 0.7308712601661682}


In [15]:
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
import numpy as np
import torch

predictions = trainer.predict(dataloaders=test_loader, ckpt_path="best", weights_only=False)

# predictions is a list of batch tensors — concatenate into one array
preds = torch.cat(predictions).numpy().flatten()
true_vals = test_df["pIC50"].values

rmse = np.sqrt(np.mean((preds - true_vals) ** 2))
mae = np.mean(np.abs(preds - true_vals))
r2 = r2_score(true_vals, preds)
spearman_rho, _ = spearmanr(true_vals, preds)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"Test Spearman ρ: {spearman_rho:.4f}")

Restoring states from the checkpoint path at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=30-val_loss=0.51.ckpt
Loaded model weights from the checkpoint at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=30-val_loss=0.51.ckpt


Output()

/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Test RMSE: 0.9834
Test MAE: 0.7309
Test R²: 0.4088
Test Spearman ρ: 0.6846
